# Free AI Video Dubbing MVP — Spanish / Arabic

Pipeline:

**Upload video → extract audio → timestamped Whisper transcription → NLLB translation → natural neural TTS (male/female, pitch-adjustable) → replace original audio → export MP4**

This first version produces a **dubbed video** with the original visuals and translated speech. It does not perform lip-sync.


## 1. Configure

- `TARGET_LANGUAGE`: `es` for Spanish or `ar` for Arabic.
- `WHISPER_MODEL`: `small` = faster/lighter, `medium` = better transcription, `large-v3` = highest accuracy but needs more GPU memory/time.
- `VOICE_GENDER`: `male` or `female`.
- `PITCH_ADJUST`: shift the voice's pitch, e.g. `"-15Hz"` for a deeper voice, `"+15Hz"` for a brighter one, `"+0Hz"` for no change.


In [ ]:
TARGET_LANGUAGE = "es"   # "es" = Spanish, "ar" = Arabic
WHISPER_MODEL = "medium"

VOICE_GENDER = "female"   # "male" or "female"
PITCH_ADJUST = "+0Hz"     # e.g. "-15Hz" for a deeper voice, "+15Hz" for a brighter voice

assert TARGET_LANGUAGE in {"es", "ar"}
assert VOICE_GENDER in {"male", "female"}


## 2. Helpers

Everything else in this notebook depends on `run()` and `WORK`, so they're defined **before**
any install step. `run()` always prints full stdout/stderr on failure and raises immediately, so
a broken step stops the notebook instead of silently continuing with the next cell.


In [ ]:
import subprocess, sys, os, re, math, json, shutil, textwrap
from pathlib import Path

WORK = Path("/content/dubbing_work")
WORK.mkdir(parents=True, exist_ok=True)

def run(cmd, check=True):
    """Run a command, always show its output, and fail loudly (with the full log) on error."""
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p.stdout

def pip_install(*packages, no_deps=False, quiet=True):
    cmd = [sys.executable, "-m", "pip", "install"]
    if quiet:
        cmd.append("-q")
    if no_deps:
        cmd.append("--no-deps")
    cmd += list(packages)
    run(cmd)


## 3. Install dependencies

**TTS engine switched to `edge-tts`.** MeloTTS (Spanish) and SILMA (Arabic) each only ship a
single fixed voice, have no gender choice, and — as the earlier debugging in this notebook
showed — drag in a pile of brittle, version-pinned dependencies (MeCab, fugashi, gruut,
`nemo_text_processing`, hard `numpy` pins) that keep breaking on current Colab Python versions.

`edge-tts` uses Microsoft Edge's free neural TTS service: no GPU, no model downloads, no pinned
legacy packages — just `pip install edge-tts`. It has dozens of natural-sounding neural voices
per language, clearly labeled male/female, plus built-in `rate` (speed) and `pitch` controls,
which is also what fixes the inconsistent fast/slow speech from before (see Section 5).


In [ ]:
run(["apt-get", "-qq", "update"])
run(["apt-get", "-qq", "install", "-y", "ffmpeg"])

pip_install("-U", "pip", "setuptools", "wheel")

pip_install(
    "faster-whisper",
    "transformers",
    "sentencepiece",
    "accelerate",
    "sacremoses",
    "soundfile",
    "pydub",
    "librosa",
    "scipy",
    "edge-tts",
    "nest_asyncio",
)

print("Installation finished.")


## 4. Upload your video

This opens your browser/device's native file picker so you can select a video file straight
from your computer or phone.


In [ ]:
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file was uploaded. Re-run this cell and choose a video file from your device.")

if len(uploaded) > 1:
    print("Multiple files were selected; using the first one:", list(uploaded.keys())[0])

VIDEO_PATH = next(iter(uploaded.keys()))
video_size_mb = len(uploaded[VIDEO_PATH]) / (1024 * 1024)
print(f"Input: {VIDEO_PATH} ({video_size_mb:.1f} MB)")


In [ ]:
# Extract mono 16-kHz WAV for Whisper
audio_path = WORK / "source.wav"
run([
    "ffmpeg", "-y", "-i", VIDEO_PATH,
    "-vn", "-ac", "1", "-ar", "16000",
    str(audio_path)
])
print(audio_path)


In [6]:
# 2. Transcribe with faster-whisper
from faster_whisper import WhisperModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print("Device:", device, "| compute:", compute_type)

whisper = WhisperModel(
    WHISPER_MODEL,
    device=device,
    compute_type=compute_type
)

segments_iter, info = whisper.transcribe(
    str(audio_path),
    beam_size=5,
    vad_filter=True,
    condition_on_previous_text=False
)

segments = list(segments_iter)
source_lang = info.language

rows = []
for s in segments:
    t = s.text.strip()
    if t:
        rows.append({
            "start": float(s.start),
            "end": float(s.end),
            "text": t
        })

json.dump(rows, open(WORK/"source_segments.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)

print("Detected language:", source_lang)
print("Segments:", len(rows))
for r in rows[:15]:
    print(f"[{r['start']:.2f}-{r['end']:.2f}] {r['text']}")


Device: cpu | compute: int8


KeyboardInterrupt: 

## 3. Translate timestamped segments

NLLB-200 supports 200 languages. The 600M distilled model is convenient for Colab, but its Hugging Face model card is **CC-BY-NC**, so treat this particular model as a prototype/research component rather than a commercial SaaS dependency unless you obtain suitable licensing.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

NLLB_MODEL = "facebook/nllb-200-distilled-600M"

# Common Whisper -> NLLB language mapping.
# Add more entries later as needed.
NLLB_CODES = {
    "en": "eng_Latn",
    "es": "spa_Latn",
    "fr": "fra_Latn",
    "de": "deu_Latn",
    "it": "ita_Latn",
    "pt": "por_Latn",
    "nl": "nld_Latn",
    "ru": "rus_Cyrl",
    "uk": "ukr_Cyrl",
    "tr": "tur_Latn",
    "pl": "pol_Latn",
    "ar": "arb_Arab",
    "fa": "pes_Arab",
    "ur": "urd_Arab",
    "hi": "hin_Deva",
    "bn": "ben_Beng",
    "ja": "jpn_Jpan",
    "ko": "kor_Hang",
    "zh": "zho_Hans",
}

target_code = "spa_Latn" if TARGET_LANGUAGE == "es" else "arb_Arab"
source_code = NLLB_CODES.get(source_lang)

if source_code is None:
    raise ValueError(
        f"No NLLB mapping configured for Whisper language '{source_lang}'. "
        "Add its NLLB code to NLLB_CODES."
    )

translation_device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL)
translator = AutoModelForSeq2SeqLM.from_pretrained(
    NLLB_MODEL,
    torch_dtype=torch.float16 if translation_device == "cuda" else torch.float32
).to(translation_device)

tokenizer.src_lang = source_code

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(translation_device)

    with torch.no_grad():
        out = translator.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_code),
            max_length=512,
            num_beams=4
        )

    return tokenizer.batch_decode(out, skip_special_tokens=True)

translated = []
BATCH = 8

for i in range(0, len(rows), BATCH):
    batch = rows[i:i+BATCH]
    texts = [x["text"] for x in batch]
    out = translate_batch(texts)

    for original, tr in zip(batch, out):
        item = dict(original)
        item["translation"] = tr.strip()
        translated.append(item)

json.dump(
    translated,
    open(WORK/"translated_segments.json","w",encoding="utf-8"),
    ensure_ascii=False, indent=2
)

for x in translated[:15]:
    print(f"[{x['start']:.2f}-{x['end']:.2f}] {x['translation']}")


## 4. Choose a voice

Lists the **live** catalog of neural voices Microsoft currently offers for `TARGET_LANGUAGE`, so
you always see real, working voice names rather than a stale hardcoded list. `DEFAULT_VOICES`
picks one sensible default per language/gender; override `VOICE_NAME` yourself with any
`ShortName` printed below (different accent, etc.) if you want something else.


In [ ]:
import asyncio
import nest_asyncio
import edge_tts

nest_asyncio.apply()

DEFAULT_VOICES = {
    ("es", "female"): "es-ES-ElviraNeural",
    ("es", "male"):   "es-ES-AlvaroNeural",
    ("ar", "female"): "ar-SA-ZariyahNeural",
    ("ar", "male"):   "ar-SA-HamedNeural",
}

async def list_language_voices(lang_prefix):
    voices = await edge_tts.list_voices()
    return [v for v in voices if v["Locale"].startswith(lang_prefix)]

available = asyncio.run(list_language_voices(TARGET_LANGUAGE))
print(f"Available {TARGET_LANGUAGE} voices:")
for v in available:
    print(f"  {v['ShortName']:<28} {v['Gender']:<7} {v['Locale']}")

VOICE_NAME = DEFAULT_VOICES[(TARGET_LANGUAGE, VOICE_GENDER)]
print("\nUsing voice:", VOICE_NAME, "| pitch:", PITCH_ADJUST)


## 5. Generate translated speech

Each segment is synthesized in two passes instead of one:

1. Generate at normal speaking rate and measure how long it actually came out.
2. If that's meaningfully longer/shorter than the original segment's timestamp window,
   **regenerate at an adjusted speaking `rate`** (edge-tts's own parameter) instead of
   mechanically time-stretching the audio afterwards.

Speaking a little faster or slower is what a real speaker does to fit a time budget — mechanical
audio stretching is what was making some segments sound unnaturally fast/slow before. Any small
leftover mismatch is just padded with silence or trimmed, never stretched.


In [ ]:
import soundfile as sf
from pydub import AudioSegment

TTS_DIR = WORK / "tts_segments"
TTS_DIR.mkdir(exist_ok=True)

def clean_tts_text(text):
    # Remove things that often cause TTS problems.
    text = re.sub(r"\s+", " ", text).strip()
    return text

async def synth(text, out_path, rate="+0%"):
    communicate = edge_tts.Communicate(text, VOICE_NAME, rate=rate, pitch=PITCH_ADJUST)
    await communicate.save(str(out_path))

def synth_to_file(text, out_path, rate="+0%"):
    asyncio.run(synth(text, out_path, rate=rate))

def clamp(value, lo, hi):
    return max(lo, min(hi, value))

generated = []

for i, item in enumerate(translated):
    text = clean_tts_text(item["translation"])
    if not text:
        continue

    target_ms = max(100, int((item["end"] - item["start"]) * 1000))
    out_wav = TTS_DIR / f"{i:05d}.wav"
    draft_wav = TTS_DIR / f"{i:05d}_draft.wav"

    # Pass 1: normal speaking rate.
    synth_to_file(text, draft_wav, rate="+0%")
    draft_audio = AudioSegment.from_file(draft_wav)
    draft_ms = len(draft_audio)

    # Pass 2: regenerate at an adjusted rate only if noticeably off-target.
    ratio = draft_ms / target_ms if target_ms else 1.0
    if 0.9 <= ratio <= 1.1:
        final_audio = draft_audio
    else:
        rate_pct = clamp((ratio - 1.0) * 100, -40, 40)
        rate_str = f"{rate_pct:+.0f}%"
        synth_to_file(text, out_wav, rate=rate_str)
        final_audio = AudioSegment.from_file(out_wav)

    # Gentle final correction: pad with silence or trim trailing silence only
    # (no pitch/tempo-altering stretch).
    if len(final_audio) < target_ms:
        final_audio += AudioSegment.silent(duration=target_ms - len(final_audio))
    elif len(final_audio) > target_ms:
        final_audio = final_audio[:target_ms]

    final_audio.export(out_wav, format="wav")
    draft_wav.unlink(missing_ok=True)

    item2 = dict(item)
    item2["tts_file"] = str(out_wav)
    generated.append(item2)

json.dump(
    generated,
    open(WORK/"generated_segments.json","w",encoding="utf-8"),
    ensure_ascii=False, indent=2
)

print("Generated:", len(generated), "speech segments using voice", VOICE_NAME)


## 6. Put each translated speech segment back on the original timeline

Each generated clip is already trimmed/padded to its target duration (Section 5), so this step
just lays each one down at its original start time and extends the timeline as needed — no
time-stretching here either.


In [ ]:
timeline = AudioSegment.silent(duration=0)

for item in generated:
    start_ms = int(item["start"] * 1000)
    speech = AudioSegment.from_wav(item["tts_file"])

    # Extend the timeline to cover the full span this segment needs (start_ms + len(speech)),
    # not just up to start_ms -- otherwise overlay() silently drops audio that falls past the
    # current length, which is what caused the dubbed track to come out completely silent before.
    end_needed_ms = start_ms + len(speech)
    if len(timeline) < end_needed_ms:
        timeline += AudioSegment.silent(duration=end_needed_ms - len(timeline))

    timeline = timeline.overlay(speech, position=start_ms)

dubbed_audio = WORK / "dubbed_audio.wav"
timeline.export(dubbed_audio, format="wav")
print("Dubbed timeline:", dubbed_audio)


In [ ]:
# 7. Create final MP4 with translated speech.
# We intentionally remove the original audio in this MVP.
OUTPUT = WORK / f"dubbed_{TARGET_LANGUAGE}.mp4"

run([
    "ffmpeg", "-y",
    "-i", VIDEO_PATH,
    "-i", str(dubbed_audio),
    "-map", "0:v:0",
    "-map", "1:a:0",
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    "-preset", "veryfast",
    "-crf", "20",
    "-c:a", "aac",
    "-b:a", "192k",
    "-shortest",
    str(OUTPUT)
])

print("DONE:", OUTPUT)


In [ ]:
# Download final video
from google.colab import files
files.download(str(OUTPUT))


## 8. Optional: export the translated transcript

The JSON file contains timestamps, original text and translated text. This is useful later for automatic clipping, captions, search, and editing.


In [ ]:
print(json.dumps(translated[:10], ensure_ascii=False, indent=2))
print("Transcript:", WORK/"translated_segments.json")


## Production roadmap

1. **Upload/URL ingestion**
2. **Scene/shot detection**
3. **Whisper word timestamps**
4. **Automatic highlight detection**
5. **Translation**
6. **Voice selection / authorized voice clone**
7. **TTS**
8. **Duration alignment**
9. **Optional lip-sync**
10. **Burned-in Spanish/Arabic subtitles**
11. **9:16 / 1:1 / 16:9 rendering**
12. **Queue workers + object storage**
13. **Download/publish to social platforms**

For commercial deployment, keep model licenses separate from the application code and require explicit authorization before cloning a person's voice.
